In [1]:
import pandas as pd
import numpy as np
import json

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("data\\raw\\")

pd.set_option('display.max_rows', None)      # все строки
pd.set_option('display.max_columns', None)   # все столбцы
pd.set_option('display.width', None)         # не разрывать строки по ширине консоли
pd.set_option('display.max_colwidth', None)  # не обрезать текст внутри ячеек
pd.set_option('display.expand_frame_repr', False) # не переносить широкие таблицы на новые строки

# Чтение файлов 

In [2]:
# award_badges даже не читаем - она не содержит никакой полезной инфы (кроме того, что награда №1 - для олимпиадников)

FILES = {
    "users_courses": "users_courses.csv",
    "user_answers": "user_answers.csv",
    "user_activity_histories": "user_activity_histories.csv",
    "user_trainings": "user_trainings.csv",
    "wk_users_courses_actions": "wk_users_courses_actions.csv",
    "lessons": "lessons.csv",
    "user_lessons": "user_lessons.csv",
    "wk_media_view_sessions": "wk_media_view_sessions.csv",
    "user_access_histories": "user_access_histories.csv",
    "user_award_badges": "user_award_badges.csv",
    "users": "users.csv"
}

DATE_COLS = {
    "users_courses": ["access_finished_at", "wk_officially_started_at", "wk_course_completed_at", "created_at", "updated_at"],
    "user_answers": ["submitted_at"],
    "user_activity_histories": ["created_at"],
    "user_trainings": ["started_at", "finished_at", "mark_saved_at"],
    "wk_users_courses_actions": ["created_at", "updated_at"],
    "wk_media_view_sessions": ["started_at"],
    "user_access_histories": ["access_started_at", "access_expired_at"],
    "user_award_badges": ["created_at"],
    "users": ["created_at", "grade_changed_at", "updated_at", "d_updated_at", "remember_created_at", "current_sign_in_at", "last_sign_in_at"],
    "lessons": ["wk_attendance_tracking_disabled_at"]
}

In [3]:
def load_all_tables(data_dir: Path, files: dict, date_cols: dict):
    dfs = {}
    overview = []
    for name, filename in files.items():
        path = data_dir / filename
        if not path.exists():
            print(f'!!! Не найден файл: {path}')
            continue
        parse_dates = date_cols.get(name, [])
        df = pd.read_csv(path, index_col=0, parse_dates=parse_dates, low_memory=False)
        dfs[name] = df
        overview.append({
            'table': name,
            'rows': len(df),
            'cols': df.shape[1],
            'memory_mb': round(df.memory_usage(deep=True).sum() / (1024**2), 2),
            'parsed_dates': len(parse_dates)
        })
    overview_df = pd.DataFrame(overview).sort_values('rows', ascending=False).reset_index(drop=True)
    return dfs, overview_df

dfs, tables_overview = load_all_tables(DATA_DIR, FILES, DATE_COLS)
tables_overview

,table,rows,cols,memory_mb,parsed_dates
0,user_answers,15176182,13,6610.98,1
1,wk_users_courses_actions,12909207,7,3203.60,2
2,user_lessons,3070664,11,844.95,0
3,user_activity_histories,3031137,3,433.17,1
4,wk_media_view_sessions,852358,6,182.51,1
5,user_access_histories,667124,4,87.80,2
6,user_trainings,427628,12,141.96,3
7,users_courses,290835,12,103.34,5
8,user_award_badges,252843,3,21.22,1
9,users,95395,21,70.32,7


In [66]:
def summarize_duplicates(dfs: dict) -> pd.DataFrame:
    rows = []
    for name, df in dfs.items():
        dup_count = df.duplicated().sum()
        rows.append({
            'table': name,
            'rows': len(df),
            'duplicates': dup_count,
            'duplicates_%': round((dup_count / len(df)) * 100, 2) if len(df) else 0.0
        })
    summary = pd.DataFrame(rows).sort_values('duplicates_%', ascending=False)
    return summary

duplicate_report = summarize_duplicates(dfs)
duplicate_report

,table,rows,duplicates,duplicates_%
5,lessons,3369,1813,53.81
8,user_access_histories,667124,355372,53.27
4,wk_users_courses_actions,12909207,5490050,42.53
1,user_answers,15176182,4951291,32.63
7,wk_media_view_sessions,852358,22183,2.60
2,user_activity_histories,3031137,33312,1.10
0,users_courses,290835,0,0.00
6,user_lessons,3070664,0,0.00
3,user_trainings,427628,0,0.00
9,user_award_badges,252843,0,0.00


# Предобработка таблиц

In [67]:
# Карта колонок для конвертации в целочисленный тип
INT_COLS_MAP = {
    'users_courses': ['user_id', 'course_id'],
    'user_answers': ['user_id', 'task_id'],
    'user_activity_histories': ['user_lesson_id'],
    'user_trainings': ['user_id', 'training_id'],
    'wk_users_courses_actions': ['user_id', 'users_course_id', 'lesson_id'],
    'lessons': ['course_id'],
    'user_lessons': ['user_id', 'lesson_id', 'group_id', 'users_course_id'],
    'user_award_badges': ['user_id'],
    'wk_media_view_sessions' : ['count'],
    'users': ['d_wk_school_id', 'sign_in_count', 'grade_id', 'd_wk_municipal_id', 'd_wk_region_id']
}

# конвертация
for tbl, cols in INT_COLS_MAP.items():
    for col in cols:
        if col not in dfs[tbl].columns:
            continue
        if pd.api.types.is_numeric_dtype(dfs[tbl][col]):
            continue 
        dfs[tbl][col] = pd.to_numeric(
            dfs[tbl][col].astype(str).str.replace(',', '', regex=False),
            errors='coerce'
        ).astype('Int64') 

## users_courses

In [68]:
# 1. state → is_active (булев флаг)
dfs['users_courses']['is_active'] = dfs['users_courses']['state'].str.lower().eq('active')

# 2. Конвертация "числовых строк" в numeric
for col in ['wk_max_points', 'wk_max_task_count']:
    dfs['users_courses'][col] = pd.to_numeric(dfs['users_courses'][col], errors='coerce')

# 3. Нормализация прогресса: wk_points / wk_max_points → progress_pct
dfs['users_courses']['progress_pct'] = np.where(
    (dfs['users_courses']['wk_max_points'] > 0) & (dfs['users_courses']['wk_max_points'].notna()),
    (dfs['users_courses']['wk_points'] / dfs['users_courses']['wk_max_points'] * 100).round(2),
    np.nan 
)

dfs['users_courses'] = dfs['users_courses'].drop(columns=['state', 'wk_points']) 

## user_answers

In [69]:
# 1. Конвертация в bool (учитывает регистр, пробелы и безопасно обрабатывает NaN)
for col in ['solved', 'skipped', 'wk_partial_answer']:
    dfs['user_answers'][col] = dfs['user_answers'][col].astype(str).str.strip().str.lower().eq('true')

# 2. Нормализация попыток: attempts / max_attempts → attempts_pct
dfs['user_answers']['attempts_pct'] = np.where(
    (dfs['user_answers']['max_attempts'] > 0) & (dfs['user_answers']['max_attempts'].notna()),
    (dfs['user_answers']['attempts'] / dfs['user_answers']['max_attempts'] * 100).round(2),
    np.nan
)

# 3. Создание бинарных флагов по типу ресурса
dfs['user_answers']['is_lesson'] = dfs['user_answers']['resource_type'].eq('Lesson')
dfs['user_answers']['is_training'] = dfs['user_answers']['resource_type'].eq('Training')
dfs['user_answers']['is_hw'] = dfs['user_answers']['resource_type'].eq('Homework')

# 4. Удаляем лишние столбцы
dfs['user_answers'] = dfs['user_answers'].drop(columns=['performance', 'attempts', 'resource_type'])

In [70]:
# 1. Парсим JSON-строки в списки
dfs['user_answers']['parsed_results'] = dfs['user_answers']['results'].apply(
    lambda x: json.loads(x) if isinstance(x, str) else []
)

# 2. Раскрываем списки: каждый элемент списка → отдельная строка
dfs['user_answers'] = dfs['user_answers'].explode('parsed_results').reset_index(drop=True)

# 3. Извлекаем поля из словарей
dicts = dfs['user_answers']['parsed_results']

In [71]:
dfs['user_answers']['result_subtask_id'] = [next(iter(d.keys()), None) if isinstance(d, dict) else None for d in dicts]
dfs['user_answers']['result_points'] = [next(iter(d.values()), {}).get('points', None) if isinstance(d, dict) else None for d in dicts]
dfs['user_answers']['result_coefficient'] = [next(iter(d.values()), {}).get('coefficient', None) if isinstance(d, dict) else None for d in dicts]

dfs['user_answers']['result_subtask_id'] = dfs['user_answers']['result_subtask_id'].astype('Int64')
dfs['user_answers']['result_points'] = dfs['user_answers']['result_points'].astype('float32')
dfs['user_answers']['result_coefficient'] = dfs['user_answers']['result_coefficient'].astype('float32')

dfs['user_answers'] = dfs['user_answers'].drop(columns=['parsed_results', 'results'])

## user_activity_histories

In [72]:
# 1. Создание бинарных флагов по типу действия
dfs['user_activity_histories']['is_visit_video'] = dfs['user_activity_histories']['action'].eq('visit_video')
dfs['user_activity_histories']['is_visit_translation'] = dfs['user_activity_histories']['action'].eq('visit_translation')
dfs['user_activity_histories']['is_show_conspect'] = dfs['user_activity_histories']['action'].eq('show_conspect')

dfs['user_activity_histories'] = dfs['user_activity_histories'].drop(columns=['action'])

## user_trainings

In [73]:
dfs['user_trainings']['is_lesson_training'] = dfs['user_trainings']['type'].eq('UserTrainings::LessonTraining')
dfs['user_trainings']['is_regular_training'] = dfs['user_trainings']['type'].eq('UserTrainings::RegularTraining')
dfs['user_trainings']['is_olympiad_training'] = dfs['user_trainings']['type'].eq('UserTrainings::OlympiadTraining')

dfs['user_trainings']['is_started'] = dfs['user_trainings']['state'].eq('started')

dfs['user_trainings'] = dfs['user_trainings'].drop(columns=['type', 'state'])

## wk_users_courses_actions

In [74]:
# 1. Создание бинарных флагов по типу действия в курсе
dfs['wk_users_courses_actions']['is_visit_video'] = dfs['wk_users_courses_actions']['action'].eq('visit_video')
dfs['wk_users_courses_actions']['is_show_conspect'] = dfs['wk_users_courses_actions']['action'].eq('visit_preparation_material')
dfs['wk_users_courses_actions']['is_start_training'] = dfs['wk_users_courses_actions']['action'].eq('start_training')
dfs['wk_users_courses_actions']['is_user_answer'] = dfs['wk_users_courses_actions']['action'].eq('user_answer')
dfs['wk_users_courses_actions']['is_visit_translation'] = dfs['wk_users_courses_actions']['action'].eq('visit_translation')
dfs['wk_users_courses_actions']['is_scratch_playground_visited'] = dfs['wk_users_courses_actions']['action'].eq('scratch_playground_visited')

dfs['wk_users_courses_actions'] = dfs['wk_users_courses_actions'].drop(columns=['action', 'sourceable_id', 'lesson_id'])

## user_award_badges

In [75]:
# 1. Агрегируем даты первого получения наград по пользователям
dates_wide = dfs['user_award_badges'].pivot_table(
    index='user_id', columns='award_badge_id', values='created_at', aggfunc='min'
)
dates_wide.columns = [f'date_badge_{int(c)}' for c in dates_wide.columns]

# 2. Формируем булевы флаги достижения (True, если есть дата)
flags_wide = dates_wide.notna()
flags_wide.columns = [f'has_badge_{int(c[-1])}' for c in dates_wide.columns]

# 3. Объединяем флаги и даты в одну таблицу, возвращаем user_id в колонки
dfs['user_award_badges'] = pd.concat([flags_wide, dates_wide], axis=1).reset_index()

Пользователь не может получить бейдж 1 и 6 одновременно - он получает либо один, либо другой!

## users + user_award_badges

In [76]:
dfs['users']['user_id'] = dfs['users']['id'].copy()

dfs['users'] = dfs['users'].drop(columns=['is_teacher', 'current_sign_in_at', 'last_sign_in_at', 'grade_checked', 'xp', 'd_updated_at', 'id'])

In [77]:
# type → is_teacher (булев флаг: True для User::Agent)
dfs['users']['is_teacher'] = dfs['users']['type'].str.contains('Agent', na=False)
dfs['users'] = dfs['users'][dfs['users'].is_teacher == False].reset_index(drop=True)
dfs['users'] = dfs['users'].drop(columns=['type', 'is_teacher'])

# 1. Соединение: добавляем данные о наградах к профилям пользователей 
dfs['users'] = dfs['users'].merge(dfs['user_award_badges'], on='user_id', how='left')

# 2. Заполняем пропуски в булевых флагах: нет записи = награда не получена (False)
for col in dfs['users'].columns:
    if col.startswith('has_badge_'):
        dfs['users'][col] = dfs['users'][col].fillna(False).astype(bool)

In [78]:
del dfs['user_award_badges']

## user_access_histories

In [79]:
# 1. Создание бинарных флагов по типу активатора доступа
dfs['user_access_histories']['is_premium_access'] = dfs['user_access_histories']['activator_class'].eq('Courses::AccessActivators::PremiumAccessActivator')
dfs['user_access_histories']['is_revoke_access'] = dfs['user_access_histories']['activator_class'].eq('Courses::AccessActivators::RevokeAccessActivator')
dfs['user_access_histories']['is_standard_access'] = dfs['user_access_histories']['activator_class'].eq('Courses::AccessActivators::StandardAccessActivator')
dfs['user_access_histories']['is_change_duration_access'] = dfs['user_access_histories']['activator_class'].eq('Courses::AccessActivators::ChangeAccessDurationActivator')
dfs['user_access_histories']['is_month_premium_access'] = dfs['user_access_histories']['activator_class'].eq('Courses::AccessActivators::MonthPremiumAccessActivator')

dfs['user_access_histories'] = dfs['user_access_histories'].drop(columns=['activator_class'])

## lessons

In [80]:
def build_course_features_from_lessons(lessons: pd.DataFrame) -> pd.DataFrame:
    """
    @brief Строит агрегированные признаки по course_id на основе lessons.csv.

    Основной принцип:
    - берём только проверяемую информацию из самой таблицы lessons;
    - не считаем task_expected / conspect_expected абсолютной истиной,
      а используем их вместе с observed-признаками и mismatch-счётчиками.

    @param lessons Исходный DataFrame lessons.csv.
    @return pd.DataFrame Таблица признаков в разрезе course_id.
    """
    df = lessons.copy()

    # ------------------------------------------------------------
    # 1) Приведение типов
    # ------------------------------------------------------------
    numeric_cols = [
        "lesson_number",
        "wk_max_points",
        "wk_task_count",
        "wk_video_duration",
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    bool_cols = [
        "conspect_expected",
        "task_expected",
        "wk_survival_training_expected",
        "wk_scratch_playground_enabled",
        "wk_attendance_tracking_enabled",
    ]
    for col in bool_cols:
        df[col] = df[col].fillna(False).astype(bool)

    # ------------------------------------------------------------
    # 2) Устойчивые row-level признаки
    # ------------------------------------------------------------
    # Есть ли номер урока
    df["has_lesson_number"] = df["lesson_number"].notna()

    # Наблюдаемые задачи: считаем это более надёжным, чем один только task_expected
    df["has_tasks_observed"] = (
        df["wk_task_count"].fillna(0).gt(0)
        | df["wk_max_points"].fillna(0).gt(0)
    )

    # Есть ли видео
    df["has_video"] = df["wk_video_duration"].fillna(0).gt(0)

    # Было ли явно отключено отслеживание посещаемости
    df["attendance_disabled_marked"] = df["wk_attendance_tracking_disabled_at"].notna()

    # ------------------------------------------------------------
    # 3) Диагностические признаки качества данных
    # ------------------------------------------------------------
    # task_expected=False, но по факту задачи/баллы есть
    df["task_flag_mismatch"] = (~df["task_expected"]) & df["has_tasks_observed"]

    # Отношение "баллы на одну задачу" — полезный агрегат сложности/оценивания
    valid_task_rows = df["wk_task_count"].fillna(0).gt(0) & df["wk_max_points"].notna()
    df["points_per_task"] = np.where(
        valid_task_rows,
        df["wk_max_points"] / df["wk_task_count"],
        np.nan,
    )

    # Доля строк, где max_points ~= task_count
    df["points_equal_task"] = np.where(
        valid_task_rows,
        np.isclose(df["wk_max_points"], df["wk_task_count"], atol=1e-9),
        np.nan,
    )

    # ------------------------------------------------------------
    # 4) Агрегация по курсам
    # ------------------------------------------------------------
    course_features = (
        df.groupby("course_id", dropna=False)
        .agg(
            # Размер и структура набора уроков курса
            lessons_total_count=("course_id", "size"),
            lessons_with_number_count=("has_lesson_number", "sum"),

            # Флаги конфигурации
            lessons_with_conspect_count=("conspect_expected", "sum"),
            lessons_task_expected_rows=("task_expected", "sum"),
            lessons_attendance_enabled_rows=("wk_attendance_tracking_enabled", "sum"),
            lessons_attendance_disabled_marked_rows=("attendance_disabled_marked", "sum"),
            lessons_survival_rows=("wk_survival_training_expected", "sum"),
            lessons_scratch_rows=("wk_scratch_playground_enabled", "sum"),

            # Наблюдаемая задача / нагрузка
            lessons_rows_with_tasks_observed=("has_tasks_observed", "sum"),
            lessons_task_count_sum=("wk_task_count", "sum"),
            lessons_task_count_mean=("wk_task_count", "mean"),
            lessons_task_count_max=("wk_task_count", "max"),

            # Балльная система
            lessons_max_points_sum=("wk_max_points", "sum"),
            lessons_max_points_mean=("wk_max_points", "mean"),
            lessons_max_points_max=("wk_max_points", "max"),
            lessons_points_per_task_mean=("points_per_task", "mean"),
            lessons_points_equal_task_share=("points_equal_task", "mean"),

            # Видео
            lessons_rows_with_video=("has_video", "sum"),
            lessons_video_duration_sum=("wk_video_duration", "sum"),
            lessons_video_duration_mean=("wk_video_duration", "mean"),
            lessons_video_duration_max=("wk_video_duration", "max"),

            # Диагностика качества данных
            lessons_task_flag_mismatch_rows=("task_flag_mismatch", "sum"),
        )
        .reset_index()
    )

    # ------------------------------------------------------------
    # 5) Доли / coverage-признаки
    # ------------------------------------------------------------
    denom = course_features["lessons_total_count"].replace(0, np.nan)

    share_map = {
        "lessons_with_number_share": "lessons_with_number_count",
        "lessons_conspect_expected_share": "lessons_with_conspect_count",
        "lessons_task_expected_share": "lessons_task_expected_rows",
        "lessons_tasks_observed_share": "lessons_rows_with_tasks_observed",
        "lessons_video_share": "lessons_rows_with_video",
        "lessons_attendance_enabled_share": "lessons_attendance_enabled_rows",
        "lessons_survival_share": "lessons_survival_rows",
        "lessons_scratch_share": "lessons_scratch_rows",
        "lessons_task_flag_mismatch_share": "lessons_task_flag_mismatch_rows",
    }

    for new_col, base_col in share_map.items():
        course_features[new_col] = course_features[base_col] / denom

    return course_features.sort_values("course_id").reset_index(drop=True)

dfs["courses"] = build_course_features_from_lessons(dfs["lessons"])

In [81]:
cols = ['course_id', 'lessons_total_count', 
       
'lessons_task_count_sum', 
'lessons_task_count_mean',
'lessons_task_count_max', 
'lessons_max_points_sum',
'lessons_max_points_mean', 
'lessons_max_points_max',
'lessons_points_per_task_mean', 
'lessons_video_duration_sum',
'lessons_video_duration_mean', 
'lessons_video_duration_max',

'lessons_points_equal_task_share',
'lessons_with_number_share',
'lessons_conspect_expected_share',
'lessons_task_expected_share',
'lessons_tasks_observed_share',
'lessons_video_share',
'lessons_attendance_enabled_share', 
'lessons_survival_share',
'lessons_scratch_share', 
'lessons_task_flag_mismatch_share'
]

dfs["courses"] = dfs["courses"][cols].fillna(0)

## user_courses + courses

In [82]:
dfs["users_courses"] = dfs["users_courses"].merge(
    dfs["courses"],
    on="course_id",
    how="left"
)

dfs["users_courses"] = dfs["users_courses"][dfs["users_courses"].wk_max_points - dfs["users_courses"].lessons_max_points_sum == 0].reset_index(drop=True)

In [83]:
del dfs["courses"]
del dfs["lessons"]

In [113]:
dfs['users_courses'] = dfs['users_courses'].drop(columns=['lessons_max_points_sum', 'lessons_task_count_sum'])

# Описание таблиц

In [114]:
def eda_report(dfs: dict, max_cat_examples: int = 2):

    with pd.option_context(
        'display.max_columns', None,
        'display.width', 120,
        'display.precision', 0,
        'display.float_format', lambda x: f'{x:.0f}' if abs(x) >= 0.01 else f'{x:.1f}',
        'display.max_colwidth', 40
    ):
        
        for name, df in dfs.items():
            print(f"\n{'='*20} {name} {'='*20}\n")

            # 1. INFO
            mem_mb = df.memory_usage(deep=True).sum() / (1024**2)
            missing_total = df.isnull().sum().sum()
            missing_pct = (missing_total / (df.shape[0] * df.shape[1])) * 100

            print(f"Shape: {df.shape[0]:,} строк × {df.shape[1]} столбцов | 💾 {mem_mb:.2f} MB")
            print(f"Пропуски: {missing_total:,} ({missing_pct:.2f}% от всех ячеек)")

            # 2. ЧИСЛОВЫЕ ПРИЗНАКИ
            num_cols = df.select_dtypes(include=['number']).columns
            if not num_cols.empty:
                print(f"\nЧИСЛОВЫЕ ПРИЗНАКИ:")
                num_stats = df[num_cols].describe().T
                num_stats['missing_%'] = (df[num_cols].isnull().mean() * 100).round(0)
                display_cols = ['count', 'mean', 'std', 'min', '50%', 'max', 'missing_%']
                available_cols = [c for c in display_cols if c in num_stats.columns]
                print(num_stats[available_cols].to_string())

            # 3. БУЛЕВЫ ПРИЗНАКИ
            bool_cols = df.select_dtypes(include='bool').columns
            if not bool_cols.empty:
                print(f"\nБУЛЕВЫ ПРИЗНАКИ:")
                bool_rows = []
                for col in bool_cols:
                    nuniq = df[col].nunique(dropna=True)
                    missing = df[col].isnull().sum()
                    missing_pct = (missing / len(df)) * 100
                    
                    vc = df[col].value_counts(normalize=True)
                    top_str = " | ".join([f"'{k}' ({v:.0%})" for k, v in vc.items()])
                        
                    bool_rows.append({
                        "признак": col,
                        "уникальных": nuniq,
                        "пропуски": f"{missing} ({missing_pct:.0f}%)",
                        "распределение": top_str
                    })
                bool_df = pd.DataFrame(bool_rows)
                print(bool_df.to_string(index=False))
                
            # 3. КАТЕГОРИАЛЬНЫЕ ПРИЗНАКИ
            cat_cols = df.select_dtypes(include=['object', 'category']).columns
            if not cat_cols.empty:
                print(f"\nКАТЕГОРИАЛЬНЫЕ ПРИЗНАКИ:")
                cat_rows = []
                for col in cat_cols:
                    nuniq = df[col].nunique(dropna=True)
                    missing = df[col].isnull().sum()
                    missing_pct = (missing / len(df)) * 100
                    
                    vc = df[col].value_counts(normalize=True).head(max_cat_examples)
                    top_str = " | ".join([f"'{k}' ({v:.0%})" for k, v in vc.items()])
                    if nuniq > max_cat_examples:
                        top_str += " ..."
                        
                    cat_rows.append({
                        "признак": col,
                        "тип": str(df[col].dtype),
                        "уникальных": nuniq,
                        "пропуски": f"{missing} ({missing_pct:.0f}%)",
                        "топ-распределение": top_str
                    })
                cat_df = pd.DataFrame(cat_rows)
                print(cat_df.to_string(index=False))

            # 4. ВРЕМЕННЫЕ ДИАПАЗОНЫ
            time_cols = df.select_dtypes(include='datetime64[ns]').columns
            if not time_cols.empty:
                print(f"\nВРЕМЕННЫЕ ДИАПАЗОНЫ:")
                for col in time_cols:
                    mn, mx = df[col].min(), df[col].max()
                    delta = mx - mn
                    print(f"   • {col}: {mn} → {mx} ({delta.days} дн.)")

eda_report(dfs)


==================== users_courses ====================

Shape: 279,953 строк × 31 столбцов | 💾 67.55 MB
Пропуски: 546,473 (6.30% от всех ячеек)

ЧИСЛОВЫЕ ПРИЗНАКИ:
                                  count    mean      std    min    50%       max  missing_%
user_id                          279953  716387    26696 665740 716914    761578        0.0
course_id                        279953 5144693 24394318    754    771 170000688        0.0
wk_max_points                    279953      78       90    0.0     62       542        0.0
wk_max_viewable_lessons          279953      12        7    0.0     14        80        0.0
wk_max_task_count                279953      81       91    0.0     62       513        0.0
progress_pct                     196225      72       25    0.0     80       100         30
lessons_total_count              279953      14        8      1     16       115        0.0
lessons_task_count_mean          279953       5        3    0.0      4        23        0.0
lesson

In [120]:
dfs["users_courses"][dfs["users_courses"].user_id == 745551]

,user_id,course_id,created_at,updated_at,access_finished_at,wk_max_points,wk_max_viewable_lessons,wk_max_task_count,wk_officially_started_at,wk_course_completed_at,is_active,progress_pct,lessons_total_count,lessons_task_count_mean,lessons_task_count_max,lessons_max_points_mean,lessons_max_points_max,lessons_points_per_task_mean,lessons_video_duration_sum,lessons_video_duration_mean,lessons_video_duration_max,lessons_points_equal_task_share,lessons_with_number_share,lessons_conspect_expected_share,lessons_task_expected_share,lessons_tasks_observed_share,lessons_video_share,lessons_attendance_enabled_share,lessons_survival_share,lessons_scratch_share,lessons_task_flag_mismatch_share
148151,745551,772,2025-11-24 17:34:00,2025-11-28 06:40:00,2026-05-24,79.0,14.0,91.0,NaT,NaT,True,79.75,18,5.055556,15.0,4.388889,15.0,0.944444,88.0,6.285714,11.0,0.944444,0.0,0.777778,0.777778,1.0,0.777778,0.0,0.0,0.0,0.222222
168094,745551,763,2025-11-18 19:25:00,2025-11-28 06:44:00,2026-05-18,4.0,5.0,4.0,2025-11-13,NaT,True,68.75,5,2.000000,2.0,2.000000,2.0,1.000000,20.0,6.666667,10.0,1.000000,0.4,0.4,0.4,0.4,0.6,0.0,0.0,0.0,0.0
185278,745551,771,2025-11-21 03:10:00,2025-11-24 17:34:00,2026-05-21,63.0,14.0,63.0,NaT,NaT,True,93.65,16,3.937500,15.0,3.937500,15.0,1.000000,127.0,9.071429,14.0,1.000000,0.0,0.875,0.875,1.0,0.875,0.0,0.0,0.0,0.125
194289,745551,770,2025-11-18 19:25:00,2025-11-20 17:00:00,2026-05-18,62.0,14.0,62.0,2025-11-13,NaT,True,93.15,16,3.875000,15.0,3.875000,15.0,1.000000,140.0,10.000000,13.0,1.000000,0.0,0.875,0.875,1.0,0.875,0.0,0.0,0.0,0.125


In [ ]:
dfs["users_courses"][dfs["users_courses"].user_id == 745551]

In [4]:
dfs["user_lessons"][dfs["user_lessons"].solved_tasks_count- dfs["user_lessons"].wk_solved_tasks_count != 0]

AttributeError: 'DataFrame' object has no attribute 'wk_solved_tasks_count'

In [ ]:
dfs["user_lessons"][dfs["user_lessons"].users_course_id == 648983]

,user_id,lesson_id,group_id,video_visited,translation_visited,users_course_id,solved,solved_tasks_count,wk_points,video_viewed,wk_solved_task_count
1455550,745551,6012,6135,True,False,648983,True,2,2.0,False,2.0
1458242,745551,6002,6125,True,False,648983,True,3,3.0,False,3.0
1458485,745551,6007,6130,True,False,648983,True,3,3.0,False,3.0
1466967,745551,6004,6127,True,False,648983,True,3,3.0,False,3.0
1466983,745551,6003,6126,True,False,648983,True,3,3.0,False,3.0
1470212,745551,6005,6128,True,False,648983,True,3,3.0,False,3.0
1470577,745551,6006,6129,True,False,648983,True,3,3.0,False,3.0
1482473,745551,6008,6131,True,False,648983,True,3,2.0,False,3.0
1485753,745551,6009,6132,True,False,648983,True,5,4.0,False,5.0
1488754,745551,6010,6133,True,False,648983,True,3,3.0,False,3.0


In [116]:
dfs["user_access_histories"][dfs["user_access_histories"].users_course_id == 648983]

,users_course_id,access_started_at,access_expired_at,is_premium_access,is_revoke_access,is_standard_access,is_change_duration_access,is_month_premium_access
342291,648983,2025-11-21,2026-05-21,True,False,False,False,False
342658,648983,2025-11-21,2026-05-21,True,False,False,False,False
342699,648983,2025-11-21,2026-05-21,True,False,False,False,False
342702,648983,2025-11-21,2026-05-21,True,False,False,False,False
342872,648983,2025-11-21,2026-05-21,True,False,False,False,False
342875,648983,2025-11-21,2026-05-21,True,False,False,False,False
343012,648983,2025-11-21,2026-05-21,True,False,False,False,False
343279,648983,2025-11-21,2026-05-21,True,False,False,False,False
343280,648983,2025-11-21,2026-05-21,True,False,False,False,False
343284,648983,2025-11-21,2026-05-21,True,False,False,False,False


In [85]:
# is_teacher - удалила столбец, он был пустой
# дропаем учителей из таблиц 

In [86]:
dfs["users_courses"][dfs["users_courses"].user_id == 665808].head()

,user_id,course_id,created_at,updated_at,access_finished_at,wk_max_points,wk_max_viewable_lessons,wk_max_task_count,wk_officially_started_at,wk_course_completed_at,is_active,progress_pct,lessons_total_count,lessons_task_count_sum,lessons_task_count_mean,lessons_task_count_max,lessons_max_points_sum,lessons_max_points_mean,lessons_max_points_max,lessons_points_per_task_mean,lessons_video_duration_sum,lessons_video_duration_mean,lessons_video_duration_max,lessons_points_equal_task_share,lessons_with_number_share,lessons_conspect_expected_share,lessons_task_expected_share,lessons_tasks_observed_share,lessons_video_share,lessons_attendance_enabled_share,lessons_survival_share,lessons_scratch_share,lessons_task_flag_mismatch_share
259,665808,50000592,2025-02-19 18:01:00,2025-07-29 10:30:00,2025-06-30,329.0,31.0,328.0,NaT,NaT,True,NaN,33,328.0,9.939394,10.0,329.0,9.969697,11.0,1.00303,652.0,21.032258,31.0,0.969697,0.0,0.030303,0.939394,1.0,0.939394,0.0,0.0,0.0,0.060606


In [95]:
dfs["users_courses"].head()

,user_id,course_id,created_at,updated_at,access_finished_at,wk_max_points,wk_max_viewable_lessons,wk_max_task_count,wk_officially_started_at,wk_course_completed_at,is_active,progress_pct,lessons_total_count,lessons_task_count_sum,lessons_task_count_mean,lessons_task_count_max,lessons_max_points_sum,lessons_max_points_mean,lessons_max_points_max,lessons_points_per_task_mean,lessons_video_duration_sum,lessons_video_duration_mean,lessons_video_duration_max,lessons_points_equal_task_share,lessons_with_number_share,lessons_conspect_expected_share,lessons_task_expected_share,lessons_tasks_observed_share,lessons_video_share,lessons_attendance_enabled_share,lessons_survival_share,lessons_scratch_share,lessons_task_flag_mismatch_share
0,718902,771,2025-10-14 14:55:00,2025-11-10 14:05:00,2026-04-14,63.0,14.0,63.0,NaT,NaT,True,71.59,16,63.0,3.9375,15.0,63.0,3.9375,15.0,1.0,127.0,9.071429,14.0,1.0,0.0,0.875,0.875,1.0,0.875,0.0,0.0,0.0,0.125
1,708314,763,2025-09-19 18:04:00,2025-10-02 15:12:00,2026-03-19,4.0,5.0,4.0,2025-09-19,NaT,True,58.25,5,4.0,2.0000,2.0,4.0,2.0000,2.0,1.0,20.0,6.666667,10.0,1.0,0.4,0.4,0.4,0.4,0.6,0.0,0.0,0.0,0.0
2,703125,770,2025-09-10 18:35:00,2025-10-06 09:26:00,2026-03-10,62.0,14.0,62.0,2025-09-10,NaT,True,0.00,16,62.0,3.8750,15.0,62.0,3.8750,15.0,1.0,140.0,10.000000,13.0,1.0,0.0,0.875,0.875,1.0,0.875,0.0,0.0,0.0,0.125
3,665813,755,2025-02-19 18:57:00,2025-02-19 18:57:00,2025-06-30,5.0,0.0,1.0,NaT,NaT,True,NaN,1,1.0,1.0000,1.0,5.0,5.0000,5.0,5.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,711827,770,2025-10-07 19:25:00,2025-11-04 10:14:00,2026-04-07,62.0,14.0,62.0,2025-10-07,NaT,True,89.56,16,62.0,3.8750,15.0,62.0,3.8750,15.0,1.0,140.0,10.000000,13.0,1.0,0.0,0.875,0.875,1.0,0.875,0.0,0.0,0.0,0.125


In [ ]:
dfs["lessons"].lesson_number.unique()

array([ nan,  99., 101.,   1., 102.,  15.,  46.,  14.,  41.,  47.,  19.,
         8.,  38.,   6.,   7.,  52.,   3.,  58.,  59.,  70.,   2.,  20.,
         5.,   4.,  12., 107., 115.,  64.,  16.,  23.,  44.,  11.,  55.,
         9., 111.,  18.,  13.,  22.,  17.,  32.,  57.,  30.,  86.,  50.,
        80.,  77.,  45.,  51.,  76., 104.,  68.,  31.,  89.,  79.,  37.,
        60.,  40.,  34.,  87.,  21.,  48.,  78.,  10.,  61.,  24.,  25.,
        65.,  43.,  28.,  39.,  27., 106.,  94.,  42.,  36.,  63.,  91.,
        54., 105.,  98., 114.,  66., 108.,  72.,  75.,  67., 112.,  35.,
        33.,  56.,  62.,  90.,  26., 100.,  93.,  53.,  29.,  49.,  71.,
        88.,  73.,  74., 109.,  69.,  83.,  96.,  97.,  84., 110., 113.,
       103.,  95.,  81.,  85.,  92.,  82.])

In [ ]:
t = 170000688

In [ ]:
dfs["users_courses"][(dfs["users_courses"].course_id == t) & (dfs["users_courses"].wk_max_points ==66)].created_at

2       2025-03-18 08:44:00
3       2025-03-18 09:26:00
5       2025-03-18 18:04:00
9       2025-03-19 08:32:00
12      2025-03-25 07:01:00
16      2025-03-25 07:28:00
21      2025-03-25 08:00:00
25      2025-03-25 06:21:00
26      2025-03-25 07:42:00
28      2025-03-23 16:23:00
30      2025-03-25 22:18:00
32      2025-03-25 08:06:00
40      2025-03-24 12:24:00
41      2025-03-21 07:55:00
44      2025-02-19 18:38:00
53      2025-02-19 16:50:00
57      2025-03-05 14:22:00
60      2025-03-25 23:40:00
61      2025-03-19 12:00:00
62      2025-03-19 12:18:00
65      2025-03-24 11:13:00
67      2025-03-20 11:11:00
70      2025-04-01 05:01:00
71      2025-03-19 10:29:00
77      2025-03-19 08:53:00
78      2025-03-19 10:46:00
79      2025-03-26 03:20:00
80      2025-03-19 13:15:00
81      2025-03-26 04:35:00
86      2025-03-26 07:08:00
89      2025-02-22 20:39:00
93      2025-02-24 12:32:00
102     2025-02-24 15:35:00
108     2025-02-19 14:35:00
119     2025-03-04 15:03:00
121     2025-02-24 1

In [ ]:
dfs["lessons"][dfs["lessons"].course_id == t].wk_max_points.sum()

np.float64(17.0)

In [ ]:
dfs["user_answers"][dfs["user_answers"].user_id == 665744]

,user_id,task_id,solved,points,max_attempts,skipped,submitted_at,wk_partial_answer,async_check_status,attempts_pct,is_lesson,is_training,is_hw,result_subtask_id,result_points,result_coefficient


In [ ]:
dfs["user_trainings"][dfs["user_trainings"].user_id == 665744]

,user_id,training_id,solved_tasks_count,earned_points,submitted_answers_count,started_at,finished_at,attempts,mark,mark_saved_at,is_lesson_training,is_regular_training,is_olympiad_training,is_started


In [ ]:
dfs["user_trainings"].training_id.unique()

<IntegerArray>
[ 50002205,      2357, 170002292,      2353, 170002293,      2361,      2419,
  50002276,      2359,      2382,
 ...
      3049,      2857,      3045,      2851,      3046,      2368,      2370,
      3012,      3047,      3048]
Length: 138, dtype: Int64

In [ ]:
dfs["wk_users_courses_actions"][dfs["wk_users_courses_actions"].user_id == 665744]

,user_id,users_course_id,created_at,updated_at,is_visit_video,is_show_conspect,is_start_training,is_user_answer,is_visit_translation,is_scratch_playground_visited
352,665744,449037,2025-02-20 10:03:00,2025-02-20 10:03:00,True,False,False,False,False,False
353,665744,449093,2025-02-20 10:03:00,2025-02-20 10:03:00,True,False,False,False,False,False
6435,665744,449037,2025-02-28 14:05:00,2025-02-28 14:05:00,False,True,False,False,False,False
6620,665744,449037,2025-02-28 14:05:00,2025-02-28 14:05:00,True,False,False,False,False,False
116470,665744,458068,2025-03-24 11:02:00,2025-03-24 11:02:00,False,False,False,False,True,False
161383,665744,458068,2025-03-26 14:06:00,2025-03-26 14:06:00,False,False,False,False,True,False
3452443,665744,570853,2025-10-22 07:19:00,2025-10-22 07:19:00,True,False,False,False,False,False
3452444,665744,570853,2025-10-22 07:19:00,2025-10-22 07:19:00,True,False,False,False,False,False
3454003,665744,570853,2025-10-22 07:18:00,2025-10-22 07:18:00,True,False,False,False,False,False


In [ ]:
dfs["user_access_histories"][dfs["user_access_histories"].users_course_id == 449093]

,users_course_id,access_started_at,access_expired_at,is_premium_access,is_revoke_access,is_standard_access,is_change_duration_access,is_month_premium_access
64,449093,2025-02-17,2025-06-30,True,False,False,False,False


In [ ]:
dfs["users_courses"][dfs["users_courses"].user_id == 665744]

,user_id,course_id,created_at,updated_at,access_finished_at,wk_max_points,wk_max_viewable_lessons,wk_max_task_count,wk_officially_started_at,wk_course_completed_at,is_active,progress_pct,lessons_total_count,lessons_task_count_sum,lessons_task_count_mean,lessons_task_count_max,lessons_max_points_sum,lessons_max_points_mean,lessons_max_points_max,lessons_points_per_task_mean,lessons_video_duration_sum,lessons_video_duration_mean,lessons_video_duration_max,lessons_points_equal_task_share,lessons_with_number_share,lessons_conspect_expected_share,lessons_task_expected_share,lessons_tasks_observed_share,lessons_video_share,lessons_attendance_enabled_share,lessons_survival_share,lessons_scratch_share,lessons_task_flag_mismatch_share
958,665744,50000592,2025-02-17 07:00:00,2025-07-29 10:30:00,2025-06-30,329.0,31.0,328.0,NaT,NaT,True,NaN,33,328.0,9.939394,10.0,329.0,9.969697,11.0,1.003030,652.0,21.032258,31.0,0.969697,0.0,0.030303,0.939394,1.0,0.939394,0.0,0.0,0.0,0.060606
1802,665744,170000688,2025-02-17 16:00:00,2025-04-02 13:14:00,2025-06-30,66.0,29.0,64.0,NaT,NaT,True,0.0,9,16.0,2.000000,2.0,17.0,2.125000,3.0,1.062500,68.0,7.555556,10.0,0.875000,0.0,0.888889,0.888889,0.888889,1.0,0.0,0.0,0.0,0.0
5928,665744,932,2025-08-28 14:26:00,2025-10-08 16:00:00,2025-12-31,86.0,11.0,80.0,NaT,NaT,True,NaN,15,80.0,8.000000,10.0,86.0,8.600000,14.0,1.060000,0.0,0.000000,0.0,0.700000,0.733333,0.0,0.4,0.666667,0.0,0.0,0.0,0.0,0.266667
8379,665744,760,2025-03-24 05:05:00,2025-04-07 07:00:00,2025-06-30,109.0,23.0,100.0,NaT,NaT,True,NaN,24,100.0,5.555556,10.0,109.0,6.055556,20.0,1.000000,0.0,0.000000,0.0,0.888889,0.958333,0.333333,0.708333,0.75,0.0,0.0,0.0,0.0,0.041667
47760,665744,754,2025-02-17 07:00:00,2025-11-08 09:00:00,2025-02-16,71.0,20.0,71.0,NaT,NaT,False,NaN,21,71.0,3.550000,4.0,71.0,3.550000,4.0,1.000000,20.0,1.000000,1.0,1.000000,0.0,0.0,0.952381,0.952381,0.952381,0.0,0.0,0.0,0.0
83524,665744,931,2025-08-28 14:26:00,2025-10-08 14:00:00,2025-12-31,57.0,11.0,57.0,NaT,NaT,True,NaN,15,57.0,4.750000,8.0,57.0,4.750000,8.0,1.000000,0.0,0.000000,0.0,1.000000,0.733333,0.733333,0.6,0.8,0.0,0.0,0.0,0.0,0.266667
134023,665744,1037,2025-10-14 05:50:00,2025-11-17 07:00:00,2026-08-31,208.0,20.0,208.0,NaT,NaT,True,NaN,22,208.0,9.454545,15.0,208.0,9.454545,15.0,1.000000,20.0,1.000000,1.0,1.000000,0.0,0.909091,0.909091,1.0,0.909091,0.0,0.0,0.0,0.090909
201076,665744,991,2025-10-14 05:50:00,2025-11-21 07:55:00,2026-08-31,210.0,20.0,223.0,NaT,NaT,True,NaN,23,222.0,9.652174,15.0,209.0,9.086957,15.0,0.956522,2.0,1.000000,1.0,0.956522,0.0,0.869565,0.869565,1.0,0.086957,0.869565,0.0,0.0,0.130435


In [ ]:
                         count    mean      std    min    50%       max  missing_%
user_id                 290835  715723    27293 665740 716070    761578        0.0
course_id               290835 6916814 29739456    754    771 170000688        0.0

                 count   mean   std    min    50%    max  missing_%
users_course_id 667124 625565 68844 449032 648983 746426        0.0

IndentationError: unexpected indent (3505609963.py, line 1)

In [ ]:
dfs["user_access_histories"]["users_course_id"].unique()

array([449032, 450273, 449033, ..., 745108, 745109, 745110],
      shape=(290784,))

In [ ]:
dfs["user_answers"][dfs["user_answers"].is_lesson==True]["task_id"].unique()

<IntegerArray>
[1142908, 1142909, 1130932, 1140417, 1144250, 1130930, 1130926, 1130934,
 1141972, 1141975,
 ...
 1189593, 1189592, 1173243, 1173244, 1173229, 1205262, 1205263, 1173249,
 1173250, 1173251]
Length: 4628, dtype: Int64

In [ ]:
50000592
170000688

In [ ]:
dfs["users_courses"]["course_id"].unique()


<IntegerArray>
[      771,       763, 170000688,       770,       755,       772,       762,
       756,      1028,       951,       760,       769,  50000592,      1096,
       836,       954,       991,       990,       932,      1061,       931,
      1030,      1029,       934,       758,       759,      1033,       993,
      1040,       943,      1027,      1039,      1036,       952,       955,
       941,       841,       938,       766,       933,       765,       764,
      1026,       942,       757,       936,      1044,       949,      1052,
       754,       761,       767,      1037,       891,       838,       895,
       890,      1031,       898,       956,       953,       946,       892,
      1103,      1062,       937,       948,      1055,       897,       886,
       894,       945,      1046,      1054,      1041,      1034,      1035,
       992,      1056,       893,      1140,      1059,      1032,      1050,
      1042,       896,      1043,      1139,     